# Γ_solar Calibration — Package 21 Central Result

This notebook derives and validates the Package 21 central result:

> **Γ_solar = arctanh(η_solar) / σ = arctanh(0.03) / 2.2 ≈ 0.014**

It also places Γ_solar in the context of the full GenesisAeon CREP spectrum
and runs the benchmark suite.

## Calibration derivation

From the UTAC fixed-point condition:
$$H^* = K \cdot \tanh(\sigma \cdot \Gamma)$$

Solving for Γ with η = H*/K:
$$\Gamma = \frac{\text{arctanh}(\eta)}{\sigma}$$

For a typical X-class flare releasing η ≈ 3% of E_max:
$$\Gamma_\text{solar} = \frac{\text{arctanh}(0.03)}{2.2} \approx 0.01365$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import math
from solar_flare_utac.constants import GAMMA_SOLAR, SIGMA_CREP, ETA_XCLASS

# Verify the derivation
print('=== Package 21 Central Result Derivation ===')
print(f'σ (CREP coupling)  = {SIGMA_CREP}')
print(f'η_solar (X-class)  = {ETA_XCLASS}')
print(f'arctanh(η)         = {math.atanh(ETA_XCLASS):.6f}')
print(f'Γ_solar = arctanh(η)/σ = {GAMMA_SOLAR:.6f}')
print(f'Benchmark target   = 0.014 ± 0.005')
print(f'PASS: {abs(GAMMA_SOLAR - 0.014) < 0.005}')

In [ ]:
# Show Γ as function of η for all GenesisAeon packages
eta_values = np.linspace(0.001, 0.99, 500)
gamma_values = np.arctanh(eta_values) / SIGMA_CREP

# Package data points
packages = [
    ('Solar Flare (P21)',      0.03,   'red',       '*', 12),
    ('Cygnus X-1 Jet (P17)',   0.10,   'orange',    'o', 8),
    ('Amazon (P19)',           0.12,   'green',     's', 8),
    ('AMOC (P18)',             0.50,   'blue',      'D', 8),
    ('Brain (P20)',            0.50,   'purple',    '^', 8),
    ('BTW Sandpile (P22)',     0.58,   'brown',     'P', 8),
    ('Manna Sandpile (P22)',   0.72,   'darkgreen', 'h', 8),
    ('ERA5 Arctic (P01)',      0.99,   'navy',      'X', 8),
]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(eta_values, gamma_values, 'k-', lw=1.5, label='Γ = arctanh(η)/2.2')

for label, eta, color, marker, ms in packages:
    gamma = math.atanh(eta) / SIGMA_CREP
    ax.scatter(eta, gamma, color=color, marker=marker, s=ms**2, zorder=5,
               label=f'{label}: Γ={gamma:.3f}')

ax.set_xlabel('η (energy conversion efficiency)')
ax.set_ylabel('Γ (CREP coupling)')
ax.set_title('CREP Γ vs. η: GenesisAeon Cross-Domain Calibration Curve')
ax.legend(fontsize=7, loc='upper left')
ax.set_xlim(0, 1)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../data/fig_04_crep_calibration.png', dpi=150)
plt.show()

In [ ]:
# Run the full benchmark suite
from solar_flare_utac.benchmark import SolarBenchmark

bm = SolarBenchmark(seed=42)
summary = bm.summary()

print(f"\n=== Benchmark Results ===")
print(f"Pass rate: {summary['passed']}/{summary['total']} "
      f"({100*summary['pass_rate']:.0f}%)\n")
for line in summary['results']:
    icon = '✓' if line.startswith('[PASS]') else '✗'
    print(f'  {icon} {line}')

In [ ]:
# UTAC phase portrait: dH/dt vs H for different Gamma values
from solar_flare_utac.active_region import MagneticActiveRegion
from solar_flare_utac.reconnection import ReconnectionThreshold

H_range = np.linspace(0, 1, 300)
rc = ReconnectionThreshold()

fig, ax = plt.subplots(figsize=(9, 5))

for gamma, color, label in [
    (GAMMA_SOLAR,          'blue',   f'Γ=Γ_solar={GAMMA_SOLAR:.4f} (quiescent)'),
    (GAMMA_SOLAR * 2,      'orange', f'Γ=2×Γ_solar (slightly elevated)'),
    (GAMMA_SOLAR * 5,      'red',    f'Γ=5×Γ_solar (pre-flare)'),
]:
    ar_temp = MagneticActiveRegion(r_buildup=0.06, h_init=0.1, seed=42)
    dH = [ar_temp.dH_dt(H, rc.reconnection_rate(H, gamma)) for H in H_range]
    ax.plot(H_range, dH, color=color, lw=1.8, label=label)

ax.axhline(0, color='black', lw=0.8)
ax.axvline(0.10, ls=':', color='blue', alpha=0.5)
ax.axvline(0.60, ls=':', color='red', alpha=0.5, label='H_threshold')
ax.set_xlabel('H (normalised free energy)')
ax.set_ylabel('dH/dt [hr⁻¹]')
ax.set_title('UTAC Phase Portrait: dH/dt vs H for varying CREP Γ')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()